# DevGraph-RL — RLHF Training (Colab)

**Pipeline:** Collect → Sklearn Analysis → Keras Sweep → PyTorch DPO Training

**Requirements:**
- Runtime: GPU (T4 or better). Runtime → Change runtime type → T4 GPU
- Your `reward_history.jsonl` uploaded to this session (or pulled from GitHub)

**Run cells top to bottom. Each section is independent and can be re-run.**

## 0 — GPU Check

In [ ]:
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

## 1 — Install Dependencies

In [ ]:
%%capture
# Core ML stack
!pip install trl>=0.9.0 peft>=0.11.0 transformers>=4.41.0 datasets>=2.20.0
!pip install keras>=3.3.0 tensorflow-cpu>=2.16.0
!pip install scikit-learn>=1.5.0 numpy>=1.26.0
!pip install huggingface_hub>=0.23.0

print("Dependencies installed.")

## 2 — Clone Repo + Mount Reward Data

In [ ]:
import os
from pathlib import Path

# ── Config — edit these ──────────────────────────────────────────
GITHUB_REPO  = "https://github.com/YOUR_USERNAME/devgraph-rl.git"  # your repo
HF_TOKEN     = ""      # HuggingFace token — paste here to push model
HF_MODEL_ID  = ""      # e.g. "yourname/devgraph-rl-v1" — leave blank to skip push
BASE_MODEL   = "Qwen/Qwen2.5-0.5B"   # fits in free T4 VRAM
N_TRAIN_EPOCHS   = 3
N_SWEEP_EPOCHS   = 5
MAX_SWEEP_CONFIGS = 6
MIN_SCORE    = 0.0     # include all records for pair selection
# ─────────────────────────────────────────────────────────────────

# Clone repo
if not Path("/content/devgraph-rl").exists():
    !git clone {GITHUB_REPO} /content/devgraph-rl
else:
    !cd /content/devgraph-rl && git pull

os.chdir("/content/devgraph-rl")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Upload reward_history.jsonl from your local machine
# Option A: upload manually
from google.colab import files
import shutil

print("Upload your reward_history.jsonl file from your local machine.")
print("Location on WSL2: ~/devgraph-rl/data/vector_store/reward_history.jsonl")
print()

uploaded = files.upload()
for fname in uploaded:
    dest = Path("/content/devgraph-rl/data/vector_store") / fname
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(fname, dest)
    print(f"Saved to {dest}")

In [ ]:
# Verify reward data is present
import json

history_path = Path("/content/devgraph-rl/data/vector_store/reward_history.jsonl")
if history_path.exists():
    records = [json.loads(l) for l in history_path.read_text().strip().splitlines() if l.strip()]
    print(f"Records found: {len(records)}")
    for r in records[:3]:
        print(f"  score={r.get('final_score', 0):.3f} | task={r.get('task','')[:50]}")
else:
    print("ERROR: reward_history.jsonl not found. Upload it in the cell above.")

## 3 — Sklearn Analysis (Data Quality Gate)

In [ ]:
import sys
sys.path.insert(0, "/content/devgraph-rl")

from src.rewards.reward_store import get_reward_store
from src.training.data_collector import DataCollector
from src.training.sklearn_analyzer import SklearnAnalyzer

store     = get_reward_store()
analyzer  = SklearnAnalyzer(min_score_delta=0.15)
collector = DataCollector(
    reward_store=store,
    analyzer=analyzer,
    min_score=MIN_SCORE,
)

analyzed = collector.collect_and_analyze(min_score=MIN_SCORE)

# Distribution
dist = analyzed.distribution
print("=== Score Distribution ===")
print(f"  Total   : {dist.total}")
print(f"  High    : {dist.high}  (score >= 0.75)")
print(f"  Medium  : {dist.medium}")
print(f"  Low     : {dist.low}   (score < 0.40)")
print(f"  Mean    : {dist.mean:.3f} ± {dist.std:.3f}")
print(f"  Outliers: {len(dist.outlier_indices)}")

# Pairs
print(f"\n=== Training Pairs ===")
print(f"  Selected: {analyzed.pairs.pairs_selected}")
print(f"  Mean delta: {analyzed.pairs.mean_delta:.3f}")
for p in analyzed.pairs.pairs:
    print(f"  task={p.task!r:35} delta={p.score_delta:.3f}")

# Feature importance
print(f"\n=== Feature Importance ===")
fi = analyzed.feature_importance
for f, i in sorted(zip(fi.features, fi.importances), key=lambda x: -x[1]):
    bar = "█" * int(i * 30)
    print(f"  {f:20} {bar:30} {i:.3f}")

# Gate
print(f"\nReady for training: {analyzed.ready_for_training}")
if not analyzed.ready_for_training:
    print("STOP: Need at least 5 training pairs. Score more outputs in the Rewards tab.")

## 4 — Build HuggingFace Dataset

In [ ]:
from src.training.dataset_builder import DatasetBuilder

assert analyzed.ready_for_training, "Not enough pairs — run more reward scoring first."

builder = DatasetBuilder()
split   = builder.build_split(analyzed, eval_ratio=0.15)

print(f"Train samples : {split.n_train}")
print(f"Eval samples  : {split.n_eval}")
print(f"Mean delta    : {split.mean_delta:.3f}")
print(f"Columns       : {split.dataset_dict['train'].column_names}")
print()
print("Sample row:")
row = split.dataset_dict["train"][0]
print(f"  prompt   : {row['prompt'][:60]}")
print(f"  chosen   : {row['chosen'][:60]}")
print(f"  rejected : {row['rejected'][:60]}")
print(f"  delta    : {row['score_delta']:.3f}")

## 5 — Keras Hyperparameter Sweep (20-30 min)

In [ ]:
from src.training.keras_experiments import KerasExperiments
from pathlib import Path

ke = KerasExperiments(
    experiment_dir=Path("/content/devgraph-rl/data/experiments"),
    embedding_dim=384,
)

# Build embedding data from pairs
train_emb = KerasExperiments._pairs_to_embeddings_static(
    analyzed.pairs.pairs
) if hasattr(KerasExperiments, '_pairs_to_embeddings_static') else []

# Fallback: build embeddings inline
if not train_emb:
    import hashlib, struct
    def text_to_vec(text, dim=384):
        vec = []
        seed = text.encode()
        for i in range(dim):
            h = hashlib.md5(seed + i.to_bytes(4, 'little')).digest()
            vec.append(struct.unpack('f', h[:4])[0])
        norm = sum(x*x for x in vec)**0.5 or 1.0
        return [x/norm for x in vec]

    train_emb = [
        {
            "chosen_embedding":   text_to_vec(p.chosen_output),
            "rejected_embedding": text_to_vec(p.rejected_output),
        }
        for p in analyzed.pairs.pairs
    ]

eval_emb = train_emb[:max(1, len(train_emb)//10)]

print(f"Train embeddings: {len(train_emb)}")
print(f"Eval embeddings : {len(eval_emb)}")

# Small sweep space for free Colab (reduce if running slow)
sweep_space = {
    "learning_rate": [1e-3, 2e-4, 5e-5],
    "batch_size":    [8, 16],
    "hidden_dim":    [64, 128],
    "dropout":       [0.1],
}

print(f"\nStarting sweep — {MAX_SWEEP_CONFIGS} configs x {N_SWEEP_EPOCHS} epochs each...")
sweep = ke.run_sweep(
    train_data=train_emb,
    eval_data=eval_emb,
    sweep_space=sweep_space,
    n_epochs=N_SWEEP_EPOCHS,
    max_configs=MAX_SWEEP_CONFIGS,
)

print(f"\n=== Sweep Results ===")
print(f"  Runs      : {sweep.n_runs}")
print(f"  Best loss : {sweep.best_eval_loss:.4f}")
print(f"  Best lr   : {sweep.best_config.learning_rate}")
print(f"  Best batch: {sweep.best_config.batch_size}")
print(f"  Best hidden: {sweep.best_config.hidden_dim}")

print(f"\n  Ranked runs:")
for r in sweep.ranked_runs[:5]:
    print(f"    {r.config.experiment_id:8} eval={r.eval_loss:.4f} "
          f"lr={r.config.learning_rate} b={r.config.batch_size}")

## 6 — DPO Training (2-3 hours on T4)

In [ ]:
from src.training.torch_trainer import TorchTrainer, TrainerConfig

# Apply best Keras config to PyTorch trainer
config = TrainerConfig(
    base_model=BASE_MODEL,
    output_dir="/content/devgraph-rl/data/checkpoints",
    n_epochs=N_TRAIN_EPOCHS,
    learning_rate=sweep.best_config.learning_rate,
    per_device_train_batch_size=sweep.best_config.batch_size,
    per_device_eval_batch_size=sweep.best_config.batch_size,
    push_to_hub=bool(HF_TOKEN and HF_MODEL_ID),
    hub_model_id=HF_MODEL_ID or None,
    hub_token=HF_TOKEN or None,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    lora_r=16,
    lora_alpha=32,
)

# Validate before starting
warnings = TorchTrainer(config).validate_config()
if warnings:
    print("Config warnings:")
    for w in warnings:
        print(f"  ⚠  {w}")
else:
    print("Config OK — no warnings")

print(f"\nStarting DPO training...")
print(f"  Base model : {config.base_model}")
print(f"  Epochs     : {config.n_epochs}")
print(f"  LR         : {config.learning_rate}")
print(f"  Batch      : {config.per_device_train_batch_size}")
print(f"  LoRA rank  : {config.lora_r}")
print(f"  Push to Hub: {config.push_to_hub}")
if config.hub_model_id:
    print(f"  Hub repo   : {config.hub_model_id}")
print()

In [ ]:
# This cell takes 2-3 hours on a free T4 GPU
# DO NOT interrupt — Colab will save checkpoint to /content/devgraph-rl/data/checkpoints

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

trainer = TorchTrainer(config)
result  = trainer.train(split.dataset_dict)

print(f"\n=== Training Complete ===")
print(f"  Success       : {result.success}")
print(f"  Best loss     : {result.best_loss:.4f}")
print(f"  Final Δ reward: +{result.final_reward_delta:.3f}")
print(f"  Improved      : {result.improved}")
print(f"  Duration      : {result.total_duration_seconds/3600:.1f} hrs")
if result.hub_model_id:
    print(f"  Hub model     : https://huggingface.co/{result.hub_model_id}")
if result.error:
    print(f"  Error         : {result.error}")

print(f"\n  Epoch breakdown:")
for m in result.epoch_metrics:
    print(f"    Epoch {m.epoch}: loss={m.loss:.4f} "
          f"chosen={m.reward_chosen:.3f} "
          f"rejected={m.reward_rejected:.3f} "
          f"delta=+{m.reward_delta:.3f}")

## 7 — Download Checkpoint

In [ ]:
# Download training_result.json to your local machine
# Copy this to ~/devgraph-rl/data/checkpoints/ so the ML Lab tab can load it

from google.colab import files
import shutil

result_path = "/content/devgraph-rl/data/checkpoints/training_result.json"
if Path(result_path).exists():
    files.download(result_path)
    print("Downloaded training_result.json")
    print("Copy it to: ~/devgraph-rl/data/checkpoints/training_result.json")
    print("Then click 'Load History' in the ML Lab tab to view it.")
else:
    print("No result file found — training may have failed.")
    if result.error:
        print(f"Error: {result.error}")

## 8 — Quick Inference Test

In [ ]:
# Test the fine-tuned model with a simple prompt
# Only runs if training succeeded

if not result.success:
    print("Training failed — skipping inference test.")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    checkpoint = "/content/devgraph-rl/data/checkpoints"
    print(f"Loading fine-tuned model from {checkpoint}...")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    base      = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, checkpoint)
    model.eval()

    prompt = "Write a Python function that removes duplicates from a list:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt : {prompt}")
    print(f"\nResponse:")
    print(response[len(prompt):])

## Troubleshooting

| Problem | Fix |
|---|---|
| `CUDA out of memory` | Reduce `per_device_train_batch_size` to 2, set `lora_r=8` |
| `Not enough pairs` | Score more outputs in Rewards tab, re-upload reward_history.jsonl |
| `ImportError: trl` | Re-run cell 1 (Install Dependencies) |
| Colab disconnects | Checkpoint is saved every 100 steps — re-run from cell 6 |
| Sweep takes too long | Reduce `MAX_SWEEP_CONFIGS` to 3 or `N_SWEEP_EPOCHS` to 2 |
| Push to Hub fails | Check HF_TOKEN has write permissions to the repo |
